In [6]:
import matplotlib.patches as patches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import synergy_dataset as sd
from matplotlib.backends.backend_pdf import PdfPages

data_path = "./data/"

In [7]:
studies = pd.read_json("synergy_studies_validation.jsonl", lines=True)
studies_filtered = studies.sort_values("dataset_id").reset_index(drop=True)

recall_files = [
    "recalls_old1_nb.csv",
    "recalls_old1_svm.csv",
    "recalls_new2_nb.csv",
    "recalls_new2_svm.csv",
    "recalls_new2_mxbai_svm.csv",
    "recalls_new2_e5_svm.csv",
]
recall_types = [
    "ASR1.6 TF-IDF + NB",
    "ASR1.6 TF-IDF + SVM",
    "ASR2 TF-IDF + NB",
    "ASR2 TF-IDF + SVM",
    "ASR2 MXBAI + SVM",
    "ASR2 E5 + SVM",
]

dataset_name = "van_de_Schoot_2018"

In [8]:
def get_total_records(dataset_id):
    if dataset_id in {"Moran_2021_corrected", "Muthu_2021_corrected"}:
        return pd.read_csv(f"../src/datasets/{dataset_id}_shuffled_raw.csv").shape[0]
    else:
        return sd.Dataset(dataset_id).to_frame().shape[0]


def get_total_relevant(dataset_id):
    if dataset_id in {"Moran_2021_corrected", "Muthu_2021_corrected"}:
        return pd.read_csv(f"../src/datasets/{dataset_id}_shuffled_raw.csv")[
            "label_included"
        ].sum()
    else:
        return sd.Dataset(dataset_id).to_frame()["label_included"].sum()


total_relevant_dict = {dataset_name: get_total_relevant(dataset_name)}

In [9]:
recall_dfs = [pd.read_csv(data_path + f) for f in recall_files]

# Add metadata to each DataFrame
for i, df in enumerate(recall_dfs):
    df["dataset_name"] = studies_filtered["dataset_id"].values
    df["Model"] = recall_types[i]
    df["prior_inclusions"] = studies_filtered["prior_inclusions"].apply(len)
    df["prior_exclusions"] = studies_filtered["prior_exclusions"].apply(len)
    df["simulation_id"] = df.groupby("dataset_name").cumcount() + 1

df_all = pd.concat(recall_dfs, ignore_index=True)
df_all = df_all[df_all["dataset_name"] == dataset_name]

df_all_melted = df_all.melt(
    id_vars=[
        "dataset_name",
        "Model",
        "prior_inclusions",
        "prior_exclusions",
        "simulation_id",
    ],
    var_name="step",
    value_name="recall",
).dropna()

df_all_melted["step"] = df_all_melted["step"].astype(int)

df_all_melted["total_relevant"] = df_all_melted["dataset_name"].map(total_relevant_dict)

df_all_melted["relative_recall"] = df_all_melted["recall"] / (
    df_all_melted["total_relevant"] - df_all_melted["prior_inclusions"]
)

# Normalize step values to [0,1]
df_all_melted["relative_step"] = df_all_melted.groupby(["dataset_name", "Model"])[
    "step"
].transform(lambda x: x / x.max())

df_grouped = (
    df_all_melted.groupby(["Model", "relative_step"])["relative_recall"]
    .mean()
    .reset_index()
)

df_grouped["Model"] = pd.Categorical(
    df_grouped["Model"], categories=recall_types, ordered=True
)
df_grouped.sort_values("Model", inplace=True)

In [10]:
# Define zoomed in location + padding
x_min, x_max = 0.0, 0.2
y_min, y_max = 0.8, 1.0
x_padding = 0.05 * (x_max - x_min)
y_padding = 0.05 * (y_max - y_min)

with PdfPages(f"recall_{dataset_name}.pdf") as pdf:
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))

    # Main plot
    sns.lineplot(
        data=df_grouped,
        x="relative_step",
        y="relative_recall",
        hue="Model",
        ax=axes[0],
        legend=True,
    )

    # Padded rectangle bounds for main plot
    x_start = x_min - x_padding
    y_start = y_min - y_padding
    width = (x_max - x_min) + 2 * x_padding
    height = (y_max - y_min) + 2 * y_padding

    # Draw rectangle on main plot
    rect = patches.Rectangle(
        (x_start, y_start),
        width,
        height,
        linewidth=1,
        edgecolor="black",
        linestyle="--",
        facecolor="none",
        zorder=10,
    )
    axes[0].add_patch(rect)

    axes[0].set_xlabel("Proportion of Documents")
    axes[0].set_ylabel("Mean Recall")
    axes[0].set_title("Full View")

    # Zoomed-in plot
    sns.lineplot(
        data=df_grouped,
        x="relative_step",
        y="relative_recall",
        hue="Model",
        ax=axes[1],
        legend=False,
    )

    axes[1].set_xlim(x_min - x_padding, x_max + x_padding)
    axes[1].set_ylim(y_min - y_padding, y_max + y_padding)
    axes[1].set_xlabel("Proportion of Documents")
    axes[1].set_ylabel("Mean Recall")
    axes[1].set_title("Zoomed View")

    max_relevant = (
        total_relevant_dict[dataset_name] - 1
    )  # Subtract at least 1 inclusion prior
    total_docs = (
        get_total_records(dataset_name) - 2
    )  # subtract at least 1 inclusion and 1 exclusion
    x_perfect = max_relevant / total_docs

    x_vals = np.linspace(0, 1, 10000)
    random_y = x_vals

    perfect_y = np.piecewise(
        x_vals,
        [x_vals <= x_perfect, x_vals > x_perfect],
        [lambda x: x / x_perfect, 1.0],
    )

    for ax in axes:
        ax.plot(
            x_vals, random_y, color="gray", linestyle="--", linewidth=1, label="Random"
        )
        ax.plot(x_vals, perfect_y, color="gray", linewidth=1, label="Perfect")

    handles, labels = axes[0].get_legend_handles_labels()
    axes[0].legend(handles=handles, title="Models + Baselines")

    plt.suptitle(f"Mean Relative Recall for {dataset_name}")
    plt.tight_layout()

    pdf.savefig(fig)
    plt.savefig("02.06.2025 recall_van_de_Schoot_2018.pdf")
    plt.close(fig)